# Phi2 baseline model benchmark

In this notebook I am going to try the baseline model of Phi2 analyzing metrics to benchmark the model.

## Imports

In [16]:
import time
import torch
import psutil 
import pandas as pd

from transformers import(
    AutoTokenizer,
    AutoModelForCausalLM
)

## Configuration

In this part I am going to define the Model_ID, PROMPT and MAX_NEW_TOKENS that we are going to need for the analysis

In [17]:
MODEL_ID = "microsoft/phi-2"

PROMPT = """
You are an AI expert.

Explain quantization in large language models
for a beginner.

Use simple language and avoid code.
"""

MAX_NEW_TOKENS = 150

## Function for RAM

The function RAM is used to measure how much memory is the Python process using while the model is executing. It gets info about the actual process of Python, gets statistics about the memory of the process and gives metrics. The RSS (Resident Set Size) is the RAM memory that is occupied by the time and includes model, tensors, tokenizer, caches and variables. Finally by dividing by 1024^3 we convert the bytes into Gigabytes.

In [18]:
def get_ram_usage_gb():
    process = psutil.Process()
    memory_info = process.memory_info()
    return memory_info.rss / (1024 ** 3)

## Device Apple Silicon

In this part we are going to tell PyTorch that uses the Mac GPU if its available. MPS (Metal Performance Shaders) is the backend GPU of Apple Silicon for PyTorch. If we where using an NVIDIA processor it would be cuda. In this code we are looking if Mac soports the GPU acceleration, giving as a result a binary. If the result is a TRUE then the model is executed using GPU of Apple and Metal Backend, if FALSE uses the CPU, which could be slower. the model is then moved to Apple GPU or CPU and move the entry tensors to the same devices.

In [19]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple Silicon GPU")
else:
    device = torch.device("cpu")
    print("Using CPU")

Using Apple Silicon GPU


## Load Tokenizer

The Tokenizer is one of teh most important parts of a language model pipeline. The tokenizer acts like a translator between human languages and numerical representations, because transformers don´t understand human languages, it understands only numbers. 

In [20]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

## Load Model

In this part we load the brain of the LLM. We are getting the neuronal weights, transformer architecture, giant matrix and linguistic capacities. This is the real model.

CausalLM, which means Causal Lenguage Model is that the model predicts the next token, later it adds the token and continue to predict. This is a continous system.

From pretrained downloads the configuration, describing the number of layers, the hidden size, attention heads and vocab size. Later it builds the architecture for the transformer and downloads the weights. This files have billions of numbers , which are the knowladge of the model, including the language, reasoning, coding and facts. This weights are downloaded in memory, changing into PyTorch tensors.

The model uses torch.float16 and charges the weights in FP16, which uses less memory, inference is quicker and is perfect for modern hardware. FP16 (16 GB) reduces 50% of the memory that uses FP32 (32GB)

Finally, the model is moved to Apple GPU or CPU, depending on the case

In [21]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16
)

model.to(device)

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

PhiForCausalLM(
  (model): PhiModel(
    (embed_tokens): Embedding(51200, 2560)
    (layers): ModuleList(
      (0-31): 32 x PhiDecoderLayer(
        (self_attn): PhiAttention(
          (q_proj): Linear(in_features=2560, out_features=2560, bias=True)
          (k_proj): Linear(in_features=2560, out_features=2560, bias=True)
          (v_proj): Linear(in_features=2560, out_features=2560, bias=True)
          (dense): Linear(in_features=2560, out_features=2560, bias=True)
        )
        (mlp): PhiMLP(
          (activation_fn): NewGELUActivation()
          (fc1): Linear(in_features=2560, out_features=10240, bias=True)
          (fc2): Linear(in_features=10240, out_features=2560, bias=True)
        )
        (input_layernorm): LayerNorm((2560,), eps=1e-05, elementwise_affine=True, bias=True)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
    )
    (rotary_emb): PhiRotaryEmbedding()
    (embed_dropout): Dropout(p=0.0, inplace=False)
    (final_layernorm): LayerNorm((25

## Tokenización

The tokenization is the process to transform the text into numeric for the model to understand.

The tokenizer divides the text, looks for IDs and transforms them into tensors.

Then we have embeddings, the numbers doesen´t have meaning by them alone, so the models need embeddings to learn semantics, relations and context.

In [22]:
inputs = tokenizer(
    PROMPT,
    return_tensors="pt"
)

inputs = {k: v.to(device) for k, v in inputs.items()}

## Benchmark

After defining tokenizer (transforms text into numbers), model loading (downloads transfomer) and generator (produces text) we have to measure and answe questions like: 

    - ¿How quick the model is?

    - ¿How much memory does it uses?

    - ¿Which quality it has?

Th benchmark measures the time it takes to inference, latency (time the model takes to answer), number of tokens generated which will gave the number of tokens per second used to measure the throughput (indicates the speed of inference and the eficiency of hardware and software).

In [23]:
initial_ram = get_ram_usage_gb()

start_time = time.time()

outputs = model.generate(
    **inputs,
    max_new_tokens=MAX_NEW_TOKENS,
    do_sample=True,
    temperature=0.7,
    eos_token_id=tokenizer.eos_token_id
)

end_time = time.time()

final_ram = get_ram_usage_gb()

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In this part the model makes some operations:

    - Phase 1: Embeddings, where the model transform the tokens into vectors

    - Phase 2: Transformer Layer: The model makes some operations (Y = WX) which are multiplications of matrix.

    - Phase 3: Attention: the model compares tokens between them.

    - Phase 4: Logits: The model produces probabilities of the next token

    - Phase 5: Sampling: Elects the next token

    - Phase 6: Repeat: Autorregresive generation

All this consumes memory, compute and time, the benchmark measures that.

The temperature controls the creativity of the model:

- Low temperature -> More rigid answers

- High temperature -> More creative answers

## Decode output

In [24]:
generated_text = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False
)

print(generated_text)


You are an AI expert.

Explain quantization in large language models
for a beginner.

Use simple language and avoid code.

Try to use examples to explain.

quantization - the process of reducing the number of bits used to store and process data.

In other words, it is the process of compressing data.

This is useful in large-scale language models, as it reduces the amount of memory and processing power required to run them.

This makes it possible to train and run these models on a wider range of devices, which can be important for applications where speed and efficiency are critical.

For example, a large-scale language model could be used to analyze customer feedback on social media, in order to identify common issues and improve customer service.

The quantized model would be able to process this feedback more quickly


## Métricas

In [25]:
total_time = end_time - start_time

generated_tokens = outputs.shape[1]

tokens_per_second = generated_tokens / total_time

print(f"\\nInference Time: {total_time:.2f} sec")
print(f"Tokens/sec: {tokens_per_second:.2f}")
print(f"RAM Increase: {(final_ram - initial_ram):.2f} GB")

\nInference Time: 14.16 sec
Tokens/sec: 12.86
RAM Increase: 0.21 GB


## Save benchmark

In [26]:
results = {
    "model": "Phi-2",
    "precision": "FP16",
    "inference_time": total_time,
    "tokens_per_second": tokens_per_second,
    "ram_usage_gb": final_ram - initial_ram
}

df = pd.DataFrame([results])

df.to_csv(
    "../results/phi2_fp16_baseline.csv",
    index=False
)

print(df)

   model precision  inference_time  tokens_per_second  ram_usage_gb
0  Phi-2      FP16       14.155576          12.857125      0.210083


In [27]:
del model

import gc
gc.collect()

torch.mps.empty_cache()